# 04 · Validation confusion matrices for every model
Owner: Evaluation & Deployment Engineer (spec 3.6). Uses the validation predictions saved by each run; the test set stays locked until milestone 4.

**Colab:** Runtime → Change runtime type → **T4 GPU**. Upload the project zip to the Files panel, then Run all.

In [ ]:
# --- Setup: works on Colab and locally. Run this cell first. ---
# It finds the project folder (the one that has src/ and configs/), moves into it,
# and adds it to sys.path so that `from src ... import ...` works.
import os, sys, glob, zipfile

def is_root(path):
    """A folder is the project root if it contains both src/ and configs/."""
    return os.path.isdir(os.path.join(path, "src")) and os.path.isdir(os.path.join(path, "configs"))

def find_root():
    """Look in the current folder, its parent, /content and Google Drive (2 levels deep)."""
    here = os.getcwd()
    candidates = [here, os.path.dirname(here)]
    for base in ["/content", "/content/drive/MyDrive"]:
        candidates += glob.glob(base + "/*") + glob.glob(base + "/*/*")
    return next((c for c in candidates if is_root(c)), None)

IN_COLAB = "google.colab" in sys.modules
root = find_root()

# Colab, option 1: you uploaded the project zip to the Files panel (/content).
if root is None and IN_COLAB:
    for z in glob.glob("/content/*.zip"):
        zipfile.ZipFile(z).extractall("/content/project")
        print("unzipped", z)
    root = find_root()

# Colab, option 2: the project is in your Google Drive.
if root is None and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    root = find_root()

if root is None:
    raise FileNotFoundError("Project folder not found. Upload the project zip (or the repo zip from GitHub: Code -> Download ZIP) to the Colab Files "
                            "panel, or put the project folder in Google Drive, then run this cell again.")
os.chdir(root)
sys.path.insert(0, root)
print("project root:", root)

In [ ]:
# Download HAM10000 (about 3 GB, only the first time in a Colab session).
!pip install -q kagglehub
!python data/download.py

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from src.config import load_config
from src import data as D
from src.evaluate import plot_confusion_matrix
from src.experiment_log import CURVES_DIR, load_all_runs

cfg = load_config("configs/base.yaml")
CLASSES = cfg["data"]["classes"]
y_val = D.prepare(cfg)["val"]["label"].to_numpy()     # same seed -> same split as training
runs = load_all_runs()
keras_runs = runs[runs["epochs_run"].notna()].groupby("model")["run_id"].last()
keras_runs

In [ ]:
for model, run_id in keras_runs.items():
    proba = np.load(CURVES_DIR / f"{run_id}_val_proba.npy")
    y_pred = proba.argmax(axis=1)
    plot_confusion_matrix(y_val, y_pred, CLASSES, title=f"{model} (validation)"); plt.show()
    print(classification_report(y_val, y_pred, target_names=CLASSES, digits=3, zero_division=0))

**TODO:** which pair of classes is confused most for each model? (Expect mel ↔ nv and bkl ↔ mel.) This starts the error analysis of milestone 4.